# Evaluation

This notebook contains the code used to sample from the WSI images and split them into a training and test set. 

In [9]:
import sys
from pathlib import Path
import pyvips
import numpy as np
import cv2

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import random
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
print(project_root)
print(Path.cwd())

/home/stella
/home/stella/microglia-morph/cell_localization


## Sample tiles for background calculation

In [ ]:
output_dir = Path("./evaluation")
output_dir.mkdir(exist_ok=True)
random.seed(10)
path = "../data"
files = [f for f in Path(path).glob("./*.tiff")]
print(len(files))


def sample_tiles(num_tiles, img, img_name, patch_size, out_path, save=True):
    patches = []
    i = 1
    it = 1
    max_iterations = 35 * num_tiles
    while i <= num_tiles and it < max_iterations:
        x = random.randint(0, img.width - patch_size)
        y = random.randint(0, img.height - patch_size)
        cropped = img.crop(x, y, patch_size, patch_size)
        patch_np = np.ndarray(
            buffer=cropped.write_to_memory(),
            dtype=np.uint8,
            shape=[cropped.height, cropped.width, cropped.bands],
        )

        filename = f"{img_name}_{i}.png"
        cropped.write_to_file(str(out_path / filename))
        i += 1
        it += 1

    if i < num_tiles:
        print(f"WARNING: only {i}/{num_tiles} patches saved after {it}")

    return patches


for file in files:
    img = pyvips.Image.new_from_file(file, access="sequential")
    sample_tiles(40, img, file.name, 512, output_dir, save=False)

2


In [12]:
path = "./evaluation/no_tissue_patches/"
files = [f for f in Path(path).glob("./*.png")]
print(files)
mean = []
for file in files:
    img = pyvips.Image.new_from_file(file, access="sequential")
    patch_np = np.ndarray(
        buffer=img.write_to_memory(),
        dtype=np.uint8,
        shape=[img.height, img.width, img.bands],
    )
    mean.append(np.mean(patch_np, axis=(0, 1)))


print(np.mean(mean, axis=0))

[PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_33.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_7.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_19.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_15.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_31.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_3.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_8.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_21.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_40.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_25.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_35.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_36.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_35.png'), PosixPath('evaluation/no_tissue_patches/TPO_61EV.tiff_24.png'), PosixPath('evaluation/no_tissue_patches/TPO_60TPO.tiff_12.png'), PosixPath('evaluation/no_tissue_p

## Sample tiles from all tiff files

In [ ]:
from pathlib import Path

output_dir = Path("./evaluation")
output_dir.mkdir(exist_ok=True)

path = "/mnt/d/microglia_data"
files = [f for f in Path(path).glob("./*.tiff")]
print(len(files))

12


In [20]:
def sample_tiles(num_tiles, img, img_name, patch_size, out_path, save=True):
    patches = []
    i = 1
    it = 1
    max_iterations = 35 * num_tiles
    while i <= num_tiles and it < max_iterations:
        x = random.randint(0, img.width - patch_size)
        y = random.randint(0, img.height - patch_size)
        cropped = img.crop(x, y, patch_size, patch_size)
        patch_np = np.ndarray(
            buffer=cropped.write_to_memory(),
            dtype=np.uint8,
            shape=[cropped.height, cropped.width, cropped.bands],
        )
        if not filter_tissue(patch_np):
            if save:
                filename = f"{img_name}_{i}.png"
                cropped.write_to_file(str(out_path / filename))
            else:
                patches.append(cropped)
            i += 1
        it += 1

    if i < num_tiles:
        print(f"WARNING: only {i}/{num_tiles} patches saved after {it}")

    return patches

In [ ]:
for file in files:
    img = pyvips.Image.new_from_file(file, access="random")
    sample_tiles(30, img, file.name, 512, output_dir, save=True)

[PosixPath('/mnt/d/microglia_data/TPO_60.tiff'), PosixPath('/mnt/d/microglia_data/TPO_61_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_62_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_63_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_64_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_65_TPO.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV2.tiff'), PosixPath('/mnt/d/microglia_data/TPO_66_EV.tiff'), PosixPath('/mnt/d/microglia_data/TPO_67_TPO.tiff')]


## Sort sampled tiles into test and training set randomly

In [ ]:
from sklearn.model_selection import train_test_split
import os
import shutil

src_dir = Path("./evaluation")
train_dir = src_dir / "train"
test_dir = src_dir / "test"
train_dir.mkdir(exist_ok=True)
test_dir.mkdir(exist_ok=True)
files = [f for f in src_dir.iterdir() if f.is_file()]
train, test = train_test_split(files, test_size=0.3, random_state=36)

In [23]:
for file in train:
    shutil.copy(file, train_dir / file.name)

In [24]:
for file in test:
    shutil.copy(file, test_dir / file.name)